# Twisting brackets: Dorfman → the Nambu–Poisson tilde calculus, and the exceptional Ψ_Π rotation

This notebook is written for a reader who knows neither the mathematics nor
this code base. It follows one recipe from the paper
[arXiv:2409.11973, §7] and applies it twice.

**The objects.** A *generalized vector* is a pair (or triple) of geometric
objects living on a manifold: a vector field together with a differential
form. The space of such pairs is written `E = A ⊕ Z` — `A` for the vector
part, `Z` for the form part. On `E` there is a *bracket*: a rule
`[e₁, e₂]` that combines two generalized vectors into a third one
(the *Dorfman bracket* is the standard example).

**The recipe (eq. 7.3 of the paper).** Take any invertible linear map
`Ψ : E → E` and define a *new* bracket by

    [e₁, e₂]_Ψ  :=  Ψ⁻¹ [ Ψe₁, Ψe₂ ].

Nothing is assumed about `Ψ` beyond invertibility — but for specific
matrices `Ψ` the new bracket turns out to be a known, interesting structure.

**What we do.**

1. **Part 1.** Start with the Dorfman bracket on `TM ⊕ T*M` and
   `Ψ = (1 Π; 0 1)` with `Π` a bivector. We *prove mechanically* that the
   twisted bracket is exactly the *Nambu–Poisson tilde calculus* — the
   Koszul bracket, the tilde Lie derivative `ℒ̃`, the second action `𝒦̃` —
   and then check *every* calculus condition and compatibility condition
   of that calculus.
2. **Part 2.** Start with the *exceptional Courant bracket* on
   `TM ⊕ Λ²T*M ⊕ Λ⁵T*M` (Dorfman with two form slots plus one extra term
   `−η₂ ∧ dω₂`), rotate it with the E₆ matrix `Ψ_Π` of eqs. (4.10)–(4.14),
   separate every term of the result, and check the algebroid conditions of
   the rotated bracket.

**How to read the output.** Every claim below is *derived by the engine*:
a line `CLOSED (n steps)` means the engine reduced the difference of the two
sides to literal `0` in `n` named rewrite steps, each step citing a
definition or a declared axiom. `FAILED` means it could not — and we show
those too, because an honest failure is information (usually it says
"this needs an assumption you did not declare").

In [ ]:
import time
from jacopy.core.registry import PropertyRegistry
from jacopy.core.expr import Integer, Neg, Product, Sum
from jacopy.algebra.derivation import Act
from jacopy.central.objects import forms, functions, vector_fields
from jacopy.proof.strategies import ProofFailure

reg = PropertyRegistry()          # remembers which symbols are scalar functions etc.


def check(label, thunk):
    # Run one proof attempt; report CLOSED / FAILED with timing.
    t = time.time()
    try:
        out = thunk()
        chain = out[0] if isinstance(out, tuple) else out
        print(f"CLOSED  {label:<52} ({len(chain.steps):3d} steps, {time.time()-t:5.2f}s)")
        return chain
    except ProofFailure as exc:
        print(f"FAILED  {label:<52} -- {str(exc)[:90]}")
        return None


def show(chain, first=6, width=96):
    # Print the first few steps of a proof chain: rule name, before → after.
    for i, s in enumerate(chain.steps[:first], 1):
        print(f"  step {i}: {s.rule[:width]}")
        print(f"     before: {s.before._repr_inner()[:width]}")
        print(f"     after : {s.after._repr_inner()[:width]}")
    if len(chain.steps) > first:
        print(f"  ... {len(chain.steps) - first} more steps")

## Part 1 — Dorfman bracket twisted by `Ψ = (1 Π; 0 1)`

### 1.1 The cast

* `U, V, W` — vector fields (the `A = TM` side), `X` — a probe vector we
  evaluate 1-forms on;
* `ω, η, μ` — 1-forms (the `Z = T*M` side);
* `f, g, h` — scalar functions (`h` is a *probe*: vector-valued identities
  are checked by letting both sides act on `h`);
* `θ` — a **bivector**. Through the "sharp" map `θ♯ : T*M → TM` it sends a
  1-form to a vector field; this is the `Π` block of the matrix. In this
  code base a bivector is the order-1 member of the *Nambu structures*
  (`nambu_structure(p=1)`), which is why the variable is called `N`.

A generalized vector `U ⊕ ω` is represented simply as the Python pair
`(U, ω)`; brackets return such pairs.

In [ ]:
from jacopy.packages.poisson.nambu import nambu_structure

f, g, h = functions("f g h", registry=reg)
U, V, W, X = vector_fields("U V W X")
om, et, mu = forms("ω η μ", degree=1)
N = nambu_structure("θ", p=1)        # the bivector θ; N.sharp_vf(ω) is θ♯ω

print("θ♯ applied to ω:", N.sharp_vf(om)._repr_inner())

### 1.2 The matrix `Ψ`, written by hand

The paper's matrix acts on the pair `(vec, form)` by

    Ψ  (vec, form) = (vec + Π form, form),      Ψ⁻¹ (vec, form) = (vec − Π form, form).

There is no "matrix object" in the code base yet, so — exactly as a user
would — we write the two actions as two small functions. The library ships
the same two functions as `pi_twist` / `pi_twist_inverse`; we write them out
to show there is nothing hidden. We then let the engine confirm
`Ψ⁻¹Ψ = id` on a generic pair.

In [ ]:
from jacopy.packages.drinfeld.double import dorfman_double, nambu_double
from jacopy.packages.drinfeld.twist import twisted_dorfman, _twist_engine
from jacopy.packages.poisson.tilde import _normalized_by


def psi(vec, form):                       # Ψ = (1 Π; 0 1)
    return Sum(vec, N.sharp_vf(form)), form


def psi_inverse(vec, form):               # Ψ⁻¹ = (1 −Π; 0 1)
    return Sum(vec, Neg(N.sharp_vf(form))), form


engine1 = _twist_engine(N, reg)           # Cartan calculus + the bivector's rules
v2, f2 = psi_inverse(*psi(U, om))
print("Ψ⁻¹Ψ(U ⊕ ω) − (U ⊕ ω)  =  (",
      _normalized_by(engine1, Sum(v2, Neg(U)), reg), ",",
      _normalized_by(engine1, Sum(f2, Neg(om)), reg), ")")

# The twisted bracket of eq. (7.3): Ψ⁻¹ [Ψe₁, Ψe₂]_Dorfman, built from the
# library's Dorfman bracket and our two hand-written maps.
tw_vec, tw_form = twisted_dorfman(psi, psi_inverse, U, om, V, et)
print("\n[e₁,e₂]_Ψ vector part :", tw_vec._repr_inner())
print("[e₁,e₂]_Ψ form part   :", tw_form._repr_inner())

### 1.3 The twisted bracket *is* the Nambu–Poisson tilde calculus

The printed components are just the Dorfman formula with `U + θ♯ω`
substituted for `U`. The claim is that, after expanding the Lie derivatives
and interior products, they are exactly the operators of the *tilde
calculus* familiar from the Nambu–Poisson setting:

| twisted component / piece | equals (tilde calculus) |
| --- | --- |
| form part of `[e₁,e₂]_Ψ` | form part of the Nambu double: `[ω,η]_Koszul + ℒ_U η − ℒ_V ω + dι_V ω` |
| vector part of `[e₁,e₂]_Ψ` | vector part of the Nambu double **plus** `R′(ω,η) = [θ♯ω, θ♯η] − θ♯[ω,η]_Koszul` |
| `ℒ̃′_ω V := [θ♯ω, V] − θ♯(𝒦_V ω)` (eq. 7.25) | the tilde Lie derivative `ℒ̃_ω V` |
| `𝒦̃′_η U := [U, θ♯η] − θ♯(ℒ_U η)` (eq. 7.25) | the tilde second action `−ℒ̃_η U + d̃(ι_U η)` |
| Z-bracket `ℒ_{θ♯ω} η + 𝒦_{θ♯η} ω` (eq. 7.25) | the Koszul bracket `[ω,η]_θ` |

None of these proofs assumes anything about `θ`: the tilde calculus is
*born* from the twist for an arbitrary bivector. The only trace of "is θ a
Poisson tensor?" is the term `R′(ω,η)` in the vector part — it vanishes
exactly when `θ` satisfies the Poisson condition (the *fundamental
identity*, "FI"), and we will meet that declaration in §1.4.

In [ ]:
from jacopy.packages.drinfeld.twist import (
    prove_pi_twist_form_is_nambu, prove_pi_twist_vec_is_nambu_plus_r,
    prove_twisted_lie_tilde, prove_twisted_kappa_tilde, prove_twisted_z_bracket_is_koszul,
)

check("form part = Nambu double form part (Koszul born)",
      lambda: prove_pi_twist_form_is_nambu(N, U, om, V, et, (X,), registry=reg))
check("vector part = Nambu double vector part + R′",
      lambda: prove_pi_twist_vec_is_nambu_plus_r(N, U, om, V, et, h, registry=reg))
c = check("ℒ̃′_ω V  =  ℒ̃_ω V  (tilde Lie derivative)",
      lambda: prove_twisted_lie_tilde(N, om, V, h, registry=reg))
check("𝒦̃′_η U  =  −ℒ̃_η U + d̃ι_U η  (tilde second action)",
      lambda: prove_twisted_kappa_tilde(N, et, U, h, registry=reg))
check("twisted Z-bracket = Koszul bracket [ω,η]_θ",
      lambda: prove_twisted_z_bracket_is_koszul(N, om, et, (X,), registry=reg))

print("\nA look inside one proof (the tilde Lie derivative):")
show(c)

### 1.4 Calculus conditions

A *calculus* in the sense of the Drinfel'd-algebroid paper is a triple
`(ℒ, ι, d)` satisfying three conditions, plus two composition laws for the
second action `𝒦_V := −ℒ_V + dι_V`. We check them twice:

* on the **usual Cartan calculus** (the starting point — this is the
  sanity check that the machinery recognises the prototype);
* on the **tilde calculus** produced by the twist (conditions (D.5)–(D.7)
  of the paper's Appendix D).

The tilde conditions are the first place where the Poisson condition is
genuinely *needed*: they hold only if `θ` is Poisson. The engine does not
assume this silently — the prover takes `declare_fi=True`, records the
declared instances it used, and **fails honestly** with the surviving
`R′`-residual when the declaration is withheld. We show both.

In [ ]:
from jacopy.packages.drinfeld.calculus_conditions import (
    prove_calculus_condition_one, prove_calculus_condition_two, prove_calculus_condition_three,
    prove_kappa_kappa, prove_kappa_bracket,
)
from jacopy.packages.drinfeld.tilde_calculus import (
    prove_tilde_calculus_condition_one, prove_tilde_calculus_condition_two,
    prove_tilde_calculus_condition_three,
)

print("-- usual Cartan calculus (the prototype) --")
check("cond 1: ℒ_U ℒ_V − ℒ_V ℒ_U = ℒ_[U,V]",   lambda: prove_calculus_condition_one(U, V, mu, f, (X,), registry=reg))
check("cond 2: ℒ_U dι_W = dι_[U,W] + dι_W ℒ_U", lambda: prove_calculus_condition_two(U, W, et, f, (X,), registry=reg))
check("cond 3: ℒ_W dι_V = dι_W dι_V",          lambda: prove_calculus_condition_three(W, V, om, f, (X,), registry=reg))
check("𝒦_U 𝒦_V = −𝒦_U ℒ_V",                   lambda: prove_kappa_kappa(U, V, om, f, (X,), registry=reg))
check("𝒦_[U,V] = 𝒦_V 𝒦_U + ℒ_U 𝒦_V",           lambda: prove_kappa_bracket(U, V, om, f, (X,), registry=reg))

print("\n-- tilde calculus (declared Poisson condition) --")
check("(D.5) ℒ̃_ω ℒ̃_η − ℒ̃_η ℒ̃_ω = ℒ̃_[ω,η]",   lambda: prove_tilde_calculus_condition_one(N, om, et, W, f, h, registry=reg))
check("(D.6) ℒ̃_ω 𝒦̃_η − 𝒦̃_η ℒ̃_ω = 𝒦̃_[ω,η]",   lambda: prove_tilde_calculus_condition_two(N, om, et, W, f, h, registry=reg))
check("(D.7) ℒ̃_ω 𝒦̃_η + 𝒦̃_η 𝒦̃_ω = 𝒦̃_[ω,η]",   lambda: prove_tilde_calculus_condition_three(N, om, et, W, f, h, registry=reg))

print("\n-- the same condition WITHOUT the Poisson declaration --")
check("(D.5) with declare_fi=False",
      lambda: prove_tilde_calculus_condition_one(N, om, et, W, f, h, registry=reg, declare_fi=False))

### 1.5 Compatibility conditions and the algebroid axioms of the result

The paper's Appendix D lists what a *pair* of calculi (Cartan on `A`, tilde
on `Z`) must satisfy to form a Drinfel'd double: the *Jacobi compatibility*
conditions (D.8)–(D.13), the *metric-invariance* conditions (D.14)–(D.19),
and the *bracket-morphism* condition (4.19)/(4.20). Finally the resulting
double must itself satisfy the Courant-type axioms: symmetric part,
right-Leibniz, and the anchor being a bracket morphism.

All of them close. Two remarks for the reader:

* `slots`/probe arguments (`(X,)`, `h`, `[]`) only say *on what* an
  identity between forms or vectors is evaluated — a 1-form identity is
  checked on a vector `X`, a vector identity on a function `h`, a scalar
  identity needs nothing;
* the anchor-morphism axiom (4.19) is the one place where the Poisson
  condition enters the *double*: without the declaration the proof fails
  honestly, and the residual is again `R′`.

In [ ]:
from jacopy.packages.drinfeld.tilde_calculus import (
    prove_jacobi_compat_d8, prove_jacobi_compat_d9, prove_jacobi_compat_d10,
    prove_jacobi_compat_d11, prove_jacobi_compat_d12, prove_jacobi_compat_d13,
)
from jacopy.packages.drinfeld.metric_invariance import (
    prove_z_metric_invariance, prove_a_mixing_condition, prove_a_invariance_of_gz,
    prove_dual_mixing_condition, prove_double_metric_invariance,
)
from jacopy.packages.drinfeld.bracket_morphism import (
    prove_total_anchor_is_bracket_morphism, prove_phi_z_is_morphism,
    prove_morphism_compat_condition, prove_morphism_compat_condition_dual,
)
from jacopy.packages.drinfeld.double import (
    prove_nambu_double_symmetric_part_vec, prove_nambu_double_symmetric_part_form,
    prove_nambu_double_right_leibniz_vec, prove_nambu_double_right_leibniz_form,
)

print("-- Jacobi compatibility (D.8)-(D.13) --")
check("(D.8)",  lambda: prove_jacobi_compat_d8(N, U, et, mu, registry=reg))
check("(D.9)",  lambda: prove_jacobi_compat_d9(N, U, et, mu, (X,), registry=reg))
check("(D.10)", lambda: prove_jacobi_compat_d10(N, om, et, W, registry=reg))
check("(D.11)", lambda: prove_jacobi_compat_d11(N, om, V, W, f, h, registry=reg))
check("(D.12)", lambda: prove_jacobi_compat_d12(N, om, V, W, h, registry=reg))
check("(D.13)", lambda: prove_jacobi_compat_d13(N, mu, U, V, h, registry=reg))

print("\n-- metric invariance (D.14)-(D.19) --")
check("(D.14) Z-side invariance of g_Z",     lambda: prove_z_metric_invariance(N, om, et, mu, [], registry=reg))
check("(D.16) A-side mixing condition",      lambda: prove_a_mixing_condition(N, U, V, mu, (X,), registry=reg))
check("(D.17) A-invariance of g_Z",          lambda: prove_a_invariance_of_gz(N, U, et, mu, [], registry=reg))
check("(D.18) dual mixing condition",        lambda: prove_dual_mixing_condition(N, om, et, W, (X,), registry=reg))
check("(D.19) double-level metric invariance", lambda: prove_double_metric_invariance(N, U, om, V, et, W, mu, [], registry=reg))

print("\n-- bracket-morphism conditions (4.19)/(4.20) --")
check("(4.20)      𝒦̃_η U + θ♯(ℒ_U η) = [U, θ♯η]",   lambda: prove_morphism_compat_condition(N, U, et, h, registry=reg))
check("(4.20)-dual ℒ̃_ω V + θ♯(𝒦_V ω) = [θ♯ω, V]",  lambda: prove_morphism_compat_condition_dual(N, om, V, h, registry=reg))
check("θ♯ is a bracket morphism (needs Poisson)",  lambda: prove_phi_z_is_morphism(N, om, et, h, registry=reg))

print("\n-- Courant-type axioms of the twisted double --")
check("symmetric part, vector component",   lambda: prove_nambu_double_symmetric_part_vec(N, U, om, V, et, h, registry=reg))
check("symmetric part, form component",     lambda: prove_nambu_double_symmetric_part_form(N, U, om, V, et, (X,), registry=reg))
check("right-Leibniz, vector component",    lambda: prove_nambu_double_right_leibniz_vec(N, U, om, V, et, f, h, registry=reg))
check("right-Leibniz, form component",      lambda: prove_nambu_double_right_leibniz_form(N, U, om, V, et, f, (X,), registry=reg))
check("anchor ρ = U + θ♯ω is a bracket morphism (4.19)", lambda: prove_total_anchor_is_bracket_morphism(N, U, om, V, et, h, registry=reg))
check("   ... the same without the Poisson declaration",
      lambda: prove_total_anchor_is_bracket_morphism(N, U, om, V, et, h, registry=reg, declare_fi=False))

**Part 1 summary.** Writing `Ψ = (1 Π; 0 1)` as two four-line functions and
feeding the Dorfman bracket through eq. (7.3) produces, provably, the
Nambu–Poisson tilde calculus; every calculus condition and every
compatibility condition of that calculus closes, with the Poisson condition
entering exactly where the theory says it must — and nowhere else.

## Part 2 — the exceptional Courant bracket rotated by `Ψ_Π`

### 2.1 The bracket

Now `A = TM` and `Z = Λ²T*M ⊕ Λ⁵T*M`: a generalized vector is a triple
`U ⊕ ω₂ ⊕ ω₅` (a vector, a 2-form, a 5-form). The exceptional Courant
bracket is the Dorfman formula applied to *each* form slot, plus one extra
term that couples the two slots — the M-theory cross-term:

    vector part : [U, V]
    2-form part : ℒ_U η₂ − ι_V dω₂
    5-form part : ℒ_U η₅ − ι_V dω₅  −  η₂ ∧ dω₂          ← the extra Z-bracket term

with the two pairings `⟨e₁,e₂⟩₂ = ι_U η₂ + ι_V ω₂` (a 1-form) and
`⟨e₁,e₂⟩₅ = ι_U η₅ + ι_V ω₅ − ω₂ ∧ η₂` (a 4-form).

In [ ]:
from jacopy.central.tangent.exterior import d
from jacopy.packages.drinfeld.examples import (
    exceptional_courant_bracket, exceptional_pairing_two, exceptional_pairing_five,
    prove_exceptional_symmetric_part, prove_exceptional_right_leibniz,
)
from jacopy.packages.drinfeld.double import _ev      # evaluate a form on vector slots

om2, et2 = forms("ω₂ η₂", degree=2)
om5, et5 = forms("ω₅ η₅", degree=5)
X1, X2, X3, X4, X5 = vector_fields("X₁ X₂ X₃ X₄ X₅")
slots2, slots5 = (X1, X2), (X1, X2, X3, X4, X5)     # a 2-form is checked on 2 vectors, a 5-form on 5

vec, f2, f5 = exceptional_courant_bracket(U, om2, om5, V, et2, et5)
print("vector part:", vec._repr_inner())
print("2-form part:", f2._repr_inner())
print("5-form part:", f5._repr_inner())

### 2.2 The rotation matrix `Ψ_Π` (eqs. 4.10–4.14)

In block form for `A = TM`, `Z = Λ² ⊕ Λ⁵`:

    Ψ_Π = (1 Π; 0 1),   Π = (Π₃,  Π₆ + Π₃ ⊛ Π₃),   Ψ_Π⁻¹ = (1 −Π; 0 1),

where `Π₃` is a trivector (`Λ² → TM`), `Π₆` a 6-vector (`Λ⁵ → TM`), and
`(Π₃ ⊛ Π₃)(ω₅) := ½ Π₃(ι_{Π₃} ω₅)` — first contract the 5-form with the
trivector to a 2-form, then apply `Π₃` to that. Again there is no matrix
object; the action is one line per block, written with the library's
multivector maps. We check that the map is `C∞`-linear and that
`Ψ_Π⁻¹ Ψ_Π = id`.

In [ ]:
from jacopy.packages.drinfeld.examples import boxtimes           # (Π₃ ⊛ Π₃)(ω₅) = ½ Π₃(ι_{Π₃} ω₅)
from jacopy.packages.poisson.nambu import NambuSharpLinearityDefinition
from jacopy.packages.drinfeld.examples import (
    prove_exceptional_twist_linear, prove_exceptional_twist_inverse,
)

N3 = nambu_structure("Π₃", p=2)     # the trivector Π₃  (Λ² → TM via N3.sharp_vf)
N6 = nambu_structure("Π₆", p=5)     # the 6-vector Π₆   (Λ⁵ → TM via N6.sharp_vf)


def pi_block(a2, a5):               # Π (ω₂ ⊕ ω₅) = Π₃ω₂ + Π₆ω₅ + (Π₃⊛Π₃)ω₅
    return Sum(N3.sharp_vf(a2), N6.sharp_vf(a5), boxtimes(N3, a5))


def psi_pi(vec, a2, a5):            # Ψ_Π = (1 Π; 0 1) on triples
    return Sum(vec, pi_block(a2, a5)), a2, a5


def psi_pi_inverse(vec, a2, a5):    # Ψ_Π⁻¹ = (1 −Π; 0 1)
    return Sum(vec, Neg(pi_block(a2, a5))), a2, a5


print("Ψ_Π(U ⊕ ω₂ ⊕ ω₅) vector part:", psi_pi(U, om2, om5)[0]._repr_inner())
check("Ψ_Π is C∞-linear (vector block, ⊛ included)", lambda: prove_exceptional_twist_linear(N3, N6, U, om2, om5, f, h, registry=reg))
check("Ψ_Π⁻¹ Ψ_Π = id",                              lambda: prove_exceptional_twist_inverse(N3, N6, U, om2, om5, h, registry=reg))

### 2.3 The rotated bracket, term by term

Eq. (7.3) again: `[e₁,e₂]' = Ψ_Π⁻¹ [Ψ_Π e₁, Ψ_Π e₂]_exc`. The paper asks
for the result with *every term separated*. Two theorems do this:

* **form slots** — `form₂' = form₂ + ℒ_{ΔU} η₂ − ι_{ΔV} dω₂` and likewise
  for `form₅'`, where `ΔU = Π(ω₂ ⊕ ω₅)` is the vector shift produced by the
  rotation. So the rotated form slots are the original ones plus the
  Π-block acting through `ℒ` and `ι` — these are exactly where the
  *tilde-flavoured* terms of the exceptional Drinfel'd algebroid come from.
  The cross-term `−η₂ ∧ dω₂` is untouched (it never sees the vector slot);
* **vector slot** — `vec' = [U + ΔU, V + ΔV] − Π(form₂' ⊕ form₅')`, the
  `Ψ_Π⁻¹` block applied to the rotated forms.

In [ ]:
from jacopy.packages.generalized.exceptional_rotation import (
    rotated_exceptional_bracket, prove_rotated_form_decomposition, prove_rotated_vector_decomposition,
    _engine as exceptional_engine,
)


def rotated(U, a2, a5, V, b2, b5):
    # the user-level construction: Ψ_Π⁻¹ ∘ [·,·]_exc ∘ (Ψ_Π × Ψ_Π)
    return psi_pi_inverse(*exceptional_courant_bracket(*psi_pi(U, a2, a5), *psi_pi(V, b2, b5)))


rv, r2, r5 = rotated(U, om2, om5, V, et2, et5)
lib = rotated_exceptional_bracket(N3, N6, U, om2, om5, V, et2, et5)      # the library's construction
eng_check = exceptional_engine(N3, reg)
eng_check.register(NambuSharpLinearityDefinition(N6, reg))   # Π₆ distributes over sums
print("hand-written − library, normalised:",
      [str(_normalized_by(eng_check, Sum(a, Neg(b)), reg)) for a, b in zip((rv, r2, r5), lib)])
print("rotated 2-form part:", r2._repr_inner()[:150], "...")

check("form slots: original + Π-block through ℒ and ι", lambda: prove_rotated_form_decomposition(N3, N6, U, om2, om5, V, et2, et5, registry=reg))
check("vector slot: shifted Lie bracket − Π(rotated forms)", lambda: prove_rotated_vector_decomposition(N3, N6, U, om2, om5, V, et2, et5, registry=reg))

### 2.4 Conditions of the rotated bracket

An invertible `Ψ` transports every property of the initial bracket, with the
*transported* anchor and pairing:

    ρ'(e)  = ρ(Ψ_Π e) = U + Π(ω₂ ⊕ ω₅),          ⟨e₁,e₂⟩' = ⟨Ψ_Π e₁, Ψ_Π e₂⟩.

We check the two algebroid axioms concretely, component by component:

* **right-Leibniz** — `[e₁, f·e₂]' = f·[e₁,e₂]' + (ρ'(e₁) f)·e₂`;
* **symmetric part** — `[e₁,e₂]' + [e₂,e₁]' = Ψ_Π⁻¹ (0, d⟨Ψe₁,Ψe₂⟩₂, d⟨Ψe₁,Ψe₂⟩₅)`,
  i.e. the symmetric part of the initial bracket (which is `d` of the two
  pairings), carried back through `Ψ_Π⁻¹`.

(For the *unrotated* exceptional bracket these two axioms are library
theorems; we run them first.)

**Assembling the engine — and two rules a user has to write.** The proof
engine is a list of rewrite rules; each module ships the rules it needs, and
here we combine them by hand. The form slots close with the shipped rules.
The vector slot does not: the `Π₆` and `Π₃ ⊛ Π₃` legs put the cross-term
`η₂ ∧ dω₂` *inside* an operator's argument slot, and two facts that the
simplifier applies automatically to visible expressions are not applied
inside such slots — (i) graded commutativity of the wedge,
`α ∧ β = (−1)^{|α||β|} β ∧ α`, and (ii) pulling a scalar factor out of a
wedge factor, `(f·α) ∧ β = f·(α ∧ β)`. Both are ordinary facts about
forms; we write them as two small user rules (a `Definition` is a class with
`matches` and `rewrite`) and register them. Rule (i) only fires when every
factor has a concrete degree, so the sign it introduces is always exact.

In [ ]:
from jacopy.proof.expansion import Definition
from jacopy.core.wedge import Wedge
from jacopy.algebra.derivation import degree_of
from jacopy.central.calculus.scalars import is_scalar_function
from jacopy.central.calculus import ActExpansionDefinition, MultiEvalArgLinearityDefinition
from jacopy.packages.poisson.nambu import NambuSharpLinearityDefinition
from jacopy.packages.drinfeld.twist import WedgeSumScalarDefinition, FormProductToWedgeDefinition
from jacopy.packages.generalized.exceptional_rotation import _engine as exceptional_engine


class WedgeGradedOrder(Definition):
    # user rule (i): sort the factors of a wedge into a fixed order, with the
    # graded sign; only when every factor's degree is a concrete integer.
    name = "wedge graded commutativity (concrete degrees)"
    anchor = Wedge

    def __init__(self, registry):
        self._r = registry

    def _sorted(self, e):
        items = []
        for c in e.children:
            try:
                k = degree_of(c, self._r).as_int()
            except ValueError:
                return None
            if k is None:
                return None
            items.append((c, k))
        target = sorted(items, key=lambda it: (it[1], it[0]._repr_inner()))
        if [c for c, _ in target] == list(e.children):
            return None
        cur, sign = items[:], 0                       # bubble into place, tracking (−1)^{|α||β|}
        for pos in range(len(target)):
            j = next(i for i in range(pos, len(cur)) if cur[i] == target[pos])
            while j > pos:
                sign ^= (cur[j][1] * cur[j - 1][1]) & 1
                cur[j], cur[j - 1] = cur[j - 1], cur[j]
                j -= 1
        return sign, [c for c, _ in cur]

    def matches(self, e):
        return isinstance(e, Wedge) and self._sorted(e) is not None

    def rewrite(self, e):
        sign, kids = self._sorted(e)
        return Neg(Wedge(*kids)) if sign else Wedge(*kids)


class WedgeScalarFactor(Definition):
    # user rule (ii): (f·α) ∧ β → f·(α ∧ β) for a scalar function f.
    name = "wedge factor scalar pull-out"
    anchor = Wedge

    def __init__(self, registry):
        self._r = registry

    def _site(self, e):
        for i, c in enumerate(e.children):
            if isinstance(c, Product) and any(is_scalar_function(k, self._r) for k in c.children):
                return i
        return None

    def matches(self, e):
        return isinstance(e, Wedge) and self._site(e) is not None

    def rewrite(self, e):
        i = self._site(e)
        c = e.children[i]
        scalars = [k for k in c.children if is_scalar_function(k, self._r)]
        rest = [k for k in c.children if not is_scalar_function(k, self._r)]
        kids = list(e.children)
        kids[i] = rest[0] if len(rest) == 1 else Product(*rest)
        return Product(*scalars, Wedge(*kids))


engine2 = exceptional_engine(N3, reg)                    # Cartan + Π₃ rules + slot additivity
for rule in (NambuSharpLinearityDefinition(N6, reg),     # Π₆ is linear too
             ActExpansionDefinition(reg), MultiEvalArgLinearityDefinition(reg),
             WedgeSumScalarDefinition(reg), FormProductToWedgeDefinition(reg),
             WedgeGradedOrder(reg), WedgeScalarFactor(reg)):
    engine2.register(rule)


def closes(label, diff, probe):
    # normalise `probe(diff)` with engine2 and report whether it is literally 0
    t = time.time()
    residual = _normalized_by(engine2, probe(diff), reg)
    ok = residual == Integer(0)
    print(f"{'CLOSED' if ok else 'RESIDUAL'}  {label:<48} ({time.time()-t:5.2f}s)")
    if not ok:
        print("    residual:", residual._repr_inner()[:200])
    return ok


print("-- the unrotated exceptional bracket (library theorems) --")
check("symmetric part (2-form and 5-form slots)", lambda: prove_exceptional_symmetric_part(N3, U, om2, om5, V, et2, et5, slots2, slots5, registry=reg))
check("right-Leibniz  (2-form and 5-form slots)", lambda: prove_exceptional_right_leibniz(N3, U, om2, om5, V, et2, et5, f, slots2, slots5, registry=reg))

print("\n-- the rotated bracket: right-Leibniz with ρ'(e₁) = U + Π(ω₂ ⊕ ω₅) --")
rho_f = Act(Sum(U, pi_block(om2, om5)), f)                       # ρ'(e₁) f
lv, l2, l5 = rotated(U, om2, om5, Product(f, V), Product(f, et2), Product(f, et5))
bv, b2, b5 = rotated(U, om2, om5, V, et2, et5)
closes("vector slot", Sum(lv, Neg(Product(f, bv)), Neg(Product(rho_f, V))),   lambda e: Act(e, h))
closes("2-form slot", Sum(l2, Neg(Product(f, b2)), Neg(Product(rho_f, et2))), lambda e: _ev(e, slots2))
closes("5-form slot", Sum(l5, Neg(Product(f, b5)), Neg(Product(rho_f, et5))), lambda e: _ev(e, slots5))

print("\n-- the rotated bracket: symmetric part = Ψ_Π⁻¹(0, d⟨Ψe₁,Ψe₂⟩₂, d⟨Ψe₁,Ψe₂⟩₅) --")
cv, c2, c5 = rotated(V, et2, et5, U, om2, om5)
Ux, x2, x5 = psi_pi(U, om2, om5)
Vy, y2, y5 = psi_pi(V, et2, et5)
P2 = exceptional_pairing_two(Ux, x2, Vy, y2)
P5 = exceptional_pairing_five(Ux, x2, x5, Vy, y2, y5)
sv, s2, s5 = psi_pi_inverse(Integer(0), d(P2), d(P5))
closes("vector slot", Sum(bv, cv, Neg(sv)), lambda e: Act(e, h))
closes("2-form slot", Sum(b2, c2, Neg(s2)), lambda e: _ev(e, slots2))
closes("5-form slot", Sum(b5, c5, Neg(s5)), lambda e: _ev(e, slots5))

## What this walkthrough did — and what it did not

**Done, with today's tools.** Both twists of the paper's §7 recipe are
mechanised end to end: the Dorfman/`(1 Π; 0 1)` twist *is* the Nambu–Poisson
tilde calculus with all 3 + 2 calculus laws, the 6 Jacobi-compatibility and
5 metric-invariance conditions of Appendix D, the bracket-morphism
conditions, and the Courant-type axioms of the resulting double; the
exceptional bracket rotated by `Ψ_Π` is decomposed term by term and its
right-Leibniz and symmetric-part axioms close in all three components.
The Poisson condition was declared where the mathematics needs it and
refused everywhere else.

**Found on the way.** Preparing the vector-slot checks exposed a
soundness bug in the shared product rule: the multivector interior
`ι_{Π₃}` is a *composition* of derivations, not a derivation, and its
"no Leibniz split" flag was honoured for ordinary products but not for
wedges — `ι_{Π₃}(η₂ ∧ dω₂)` was being split. That is fixed and pinned by a
test (`tests/test_multivector_interior_wedge.py`).

**Not done here.** The Leibniz–Jacobi identity of the exceptional bracket
(and hence of its rotation) is not among today's library theorems: its
5-form component with the cross-term is a genuinely new proof. Likewise the
condition "`ρ'` is a bracket morphism" for the rotated bracket is the
exceptional analogue of the Poisson condition — its defect is the
`Π₃`/`Π₆` obstruction, to be studied as such rather than assumed.

**What a user would have wanted.** Everything above was written with pairs,
triples and hand-written maps. The things that had to be done *by hand* are
exactly the reusable pieces still missing from the code base:

1. a **block-operator matrix** type, so that `Ψ = (1 Π; 0 1)` is written as a
   matrix and its action, products and (unipotent) inverse are derived
   rather than typed;
2. a **generic axiom-suite façade** — give it a bracket, an anchor, a
   pairing and `D`, and it runs all the conditions checked above;
3. **engine auto-assembly** from the node types present in an expression
   (the `engine2` list above is the kind of knowledge a user should not
   need);
4. a first-class **generalized-section** object replacing the pair/triple
   convention;
5. a general **index antisymmetriser** for frame formulas such as
   eq. (4.13), `5 Π^{[a₁a₂a₃} Π^{a₄a₅]c}`.

The two wedge rules written in §2.4 are candidates for promotion into the
library. A second version of this walkthrough, written on top of items 1–3,
is the planned follow-up.